In [ ]:
import os
# Cell 1 derive biovolume factor

import pandas as pd
import numpy as np
from math import pi
import matplotlib.pyplot as plt
import datetime

# Load the data from the Excel file
file_path = os.path.join('data', 'Density_and_shell_mass_for_AH.xlsx')

# Read the density sheet
df_density = pd.read_excel(file_path, sheet_name='corbicula density')

# Strip whitespace from column names
df_density.columns = df_density.columns.str.strip()

# Convert dates based on dtype
if np.issubdtype(df_density['Date'].dtype, np.datetime64):
    df_density['Date_datetime'] = df_density['Date']
else:
    df_density['Date'] = pd.to_numeric(df_density['Date'], errors='coerce')
    def excel_date_to_datetime(excel_date):
        if pd.isnull(excel_date):
            return None
        return datetime.datetime(1899, 12, 30) + datetime.timedelta(days=excel_date)
    df_density['Date_datetime'] = df_density['Date'].apply(excel_date_to_datetime)

# Calculate average density
avg_density = df_density['mean density_m2'].mean()
print(f"Average density across sites: {avg_density:.2f} ind/m²")

# Read the shell mass sheet
df_shell = pd.read_excel(file_path, sheet_name='corbicula shell mass')

# Strip whitespace from shell columns
df_shell.columns = df_shell.columns.str.strip()

# Convert Clam collection based on dtype
if np.issubdtype(df_shell['Clam collection'].dtype, np.datetime64):
    df_shell['Clam collection_datetime'] = df_shell['Clam collection']
else:
    df_shell['Clam collection'] = pd.to_numeric(df_shell['Clam collection'], errors='coerce')
    def excel_date_to_datetime(excel_date):
        if pd.isnull(excel_date):
            return None
        return datetime.datetime(1899, 12, 30) + datetime.timedelta(days=excel_date)
    df_shell['Clam collection_datetime'] = df_shell['Clam collection'].apply(excel_date_to_datetime)

# Extract size category from sample (large, medium, small)
df_shell['size_category'] = df_shell['sample'].str.extract(r'([a-z]+)', expand=False)

# Convert dimensions from mm to cm (validate non-negative)
df_shell['length_cm'] = df_shell['length_mm'] / 10
df_shell['height_cm'] = df_shell['height_mm'] / 10
df_shell['width_cm'] = df_shell['width_mm'] / 10
df_shell = df_shell[(df_shell[['length_cm', 'height_cm', 'width_cm']] > 0).all(axis=1)]  # Drop invalid zeros/NaNs

# Calculate biovolume using ellipsoid formula: V = (π/6) * L * H * W (in cm³)
df_shell['biovolume_cm3'] = (pi / 6) * df_shell['length_cm'] * df_shell['height_cm'] * df_shell['width_cm']

# Calculate dry density (g/cm³) for validation against previous scaling (0.3 g/cm³)
df_shell['dry_density_g_cm3'] = df_shell['Dry total'] / df_shell['biovolume_cm3']

# Display the updated dataframe (select key columns)
print("\nShell data with calculated biovolumes and dry densities:")
display(df_shell[['sample', 'size_category', 'length_mm', 'height_mm', 'width_mm', 'Dry total', 'biovolume_cm3', 'dry_density_g_cm3']])

# Group and summarize biovolumes by size category
summary_biovolume = df_shell.groupby('size_category')['biovolume_cm3'].agg(['mean', 'std', 'min', 'max', 'count'])
print("\nSummary of biovolumes by size category (cm³):")
display(summary_biovolume)

# Overall average biovolume
avg_biovolume_cm3 = df_shell['biovolume_cm3'].mean()
print(f"\nOverall average biovolume: {avg_biovolume_cm3:.2f} cm³")

# Overall average dry density
avg_dry_density = df_shell['dry_density_g_cm3'].mean()
print(f"Overall average dry density: {avg_dry_density:.2f} g/cm³ (for updating scaling factors)")

# Estimate total biovolume for Lake Karapiro (area: 7.7 km² = 7.7e6 m²)
lake_area_m2 = 7.7e6
total_clams = avg_density * lake_area_m2
print(f"\nEstimated total clams in Lake Karapiro: {total_clams:,.0f}")
total_biovolume_m3 = total_clams * (avg_biovolume_cm3 / 1e6)
print(f"Estimated total biovolume: {total_biovolume_m3:,.2f} m³")

# Plots for publication (biovolume and dry density by size)
plt.figure(figsize=(8, 6))
df_shell.boxplot(column='biovolume_cm3', by='size_category', grid=False)
plt.title('Biovolume Distribution by Size Category')
plt.suptitle('')
plt.xlabel('Size Category')
plt.ylabel('Biovolume (cm³)')
plt.show()

plt.figure(figsize=(8, 6))
df_shell.boxplot(column='dry_density_g_cm3', by='size_category', grid=False)
plt.title('Dry Density Distribution by Size Category')
plt.suptitle('')
plt.xlabel('Size Category')
plt.ylabel('Dry Density (g/cm³)')
plt.show()

# Export processed data to new Excel for publication
with pd.ExcelWriter(os.path.join('data', 'processed_corbicula_biovolumes.xlsx')) as writer:
    df_shell.to_excel(writer, sheet_name='Shell_with_Biovolumes', index=False)
    df_density.to_excel(writer, sheet_name='Density', index=False)
    summary_biovolume.to_excel(writer, sheet_name='Biovolume_Summary')

print("\nProcessed data exported to 'processed_corbicula_biovolumes.xlsx'")

In [ ]:
import os
# Cell 2 - calculate biomineralisation, biovolumes and population metrics

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------------
# 1. Constants & Configuration (Section 2.4.3 & 2.4.4)
# ------------------------------------------------------------------
V_LAKE = 85e6           # m3 (Dynamic lake volume placeholder)
DV_DT = 5.53e-9         # m3/ind/day (Steady-state growth rate)
AREA = 7.409e6          # m2 (Colonizable benthic area)
TOTAL_SHELL_RATIO = 1.13 # g total / g shell
BULK_DENSITY = 1.15     # t/m3 (or g/cm3)
MOLAR_MASS_CA = 40.08   # g/mol
MOLAR_MASS_CACO3 = 100.09 # g/mol
EQ_WEIGHT_ALK = 50.043  # eq/mol (CaCO3 equivalent weight)

# ------------------------------------------------------------------
# 2. Data Loading & Preparation
# ------------------------------------------------------------------
# Load the dataset (Ensure the filename matches your local file)
file_path = os.path.join('data', 'Full_data_for_Corbicula_Calcs_Landscape_Updated_16_1_26.xlsx')
df = pd.read_excel(file_path)
df['Date'] = pd.to_datetime(df['Date'])

# Map flows
Q_up = df['Upstream Flow Arapuni (cumecs, m3/s)']
Q_down = df['Downstream Flow Karapiro (cumecs, m3/s)']

# Convert Concentrations (ug/L to g/m3; mg/L to g/m3)
Ca_up = df['Ca (ug/L)'] / 1000.0
Ca_down = df['Ca (ug/L).1'] / 1000.0
Alk_up = df['Alkalinity (mg/L)'] 
Alk_down = df['Alkalinity (mg/L).1']

# Depth profiles (Top = Station.4, Bottom = Station.2)
Ca_top = df['Ca (ug/L).4'] / 1000.0
Ca_bottom = df['Ca (ug/L).2'] / 1000.0
Alk_top = df['Alkalinity (mg/L).4']
Alk_bottom = df['Alkalinity (mg/L).2']

# ------------------------------------------------------------------
# 3. Removal Rate Calculations (Eq 1, 2, & 3)
# ------------------------------------------------------------------

# Eq 1: Longitudinal Ca Flux — Long. (mol/d)
df['J_rem_Ca_L'] = (Ca_up * Q_up - Ca_down * Q_down) * 86400 * (1/MOLAR_MASS_CA)

# Eq 2: Longitudinal Alk Flux — Long. (mol/d) — includes stoichiometric factor 1/2
df['J_rem_Alk_L'] = (Alk_up * Q_up - Alk_down * Q_down) * 86400 * (1/EQ_WEIGHT_ALK) * (1/2)

# Eq 3: Vertical Depletion Flux — Vert. (mol/d) — using V/τ = Q_down × 86400
V_div_tau = Q_down * 86400
df['J_rem_Ca_V'] = (Ca_top - Ca_bottom) * V_div_tau * (1/MOLAR_MASS_CA)
df['J_rem_Alk_V'] = (Alk_top - Alk_bottom) * V_div_tau * (1/EQ_WEIGHT_ALK) * (1/2)

# ------------------------------------------------------------------
# 4. Physiological Scaling (Eq 4, 5, 6, & 7)
# ------------------------------------------------------------------
def calculate_biological_metrics(J_rem):
    # Eq 4: Mass fixed (tonnes/day)
    m_caco3 = J_rem * MOLAR_MASS_CACO3 * 1e-6
    
    # Eq 5: Biovolume formation rate (m3/day)
    V_bio = m_caco3 * TOTAL_SHELL_RATIO * (1/BULK_DENSITY)
    
    # Filter: Only positive rates aggregated for population/density
    V_bio_pos = np.maximum(V_bio, 0)
    
    # Eq 6: Population Size (N)
    N = V_bio_pos / DV_DT
    
    # Eq 7: Density (ind/m2)
    D = N / AREA
    
    return m_caco3, V_bio, D

# Apply scaling to all four flux methods (Ca Long., Alk Long., Ca Vert., Alk Vert.)
for method in ['Ca_L', 'Alk_L', 'Ca_V', 'Alk_V']:
    m, v, d = calculate_biological_metrics(df[f'J_rem_{method}'])
    df[f'm_CaCO3_{method}'] = m
    df[f'V_bio_{method}'] = v
    df[f'Density_{method}'] = d

# ------------------------------------------------------------------
# 5. Output and Summary
# ------------------------------------------------------------------
print("--- Mean Population Densities (Positive Fluxes Only) ---")
density_cols = [c for c in df.columns if 'Density' in c]
print(df[density_cols].mean())

# Export results
df.to_csv(os.path.join('data', 'Revised_Corbicula_Mass_Balance.csv'), index=False)
print("\nSuccess: Full results saved to data/Revised_Corbicula_Mass_Balance.csv")

In [ ]:
import os
# Cell 3 - Examine residence time correlations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# ------------------------------------------------------------------
# 1. Load Data
# ------------------------------------------------------------------
filename = os.path.join('data', 'Revised_Corbicula_Mass_Balance.csv')
df = pd.read_csv(filename)
df['Date'] = pd.to_datetime(df['Date'])

# ------------------------------------------------------------------
# 2. Calculations
# ------------------------------------------------------------------
V_LAKE = 85e6 
flow_col = 'Downstream Flow Karapiro (cumecs, m3/s)'
df['tau_days'] = V_LAKE / (df[flow_col] * 86400)

# Calculate Gradients (Top - Bottom)
# Ca: ug/L -> g/m3 (/1000)
df['Delta_Ca'] = (df['Ca (ug/L).4'] - df['Ca (ug/L).2']) / 1000.0
# Alk: mg/L -> g/m3 (no conversion needed)
df['Delta_Alk'] = df['Alkalinity (mg/L).4'] - df['Alkalinity (mg/L).2']

# ------------------------------------------------------------------
# 3. Filter for Growth Periods (Positive Flux Only)
# ------------------------------------------------------------------
# We calculate stats independently for periods where THAT specific flux is positive
df_growth_Ca = df[df['J_rem_Ca_V'] > 0].copy()
df_growth_Alk = df[df['J_rem_Alk_V'] > 0].copy()

# ------------------------------------------------------------------
# 4. Statistical Analysis
# ------------------------------------------------------------------
def get_stats(x, y):
    slope, intercept, r, p, std_err = stats.linregress(x, y)
    return r, p

# Ca Stats
r_flux_ca, p_flux_ca = get_stats(df_growth_Ca['tau_days'], df_growth_Ca['J_rem_Ca_V'])
r_delta_ca, p_delta_ca = get_stats(df_growth_Ca['tau_days'], df_growth_Ca['Delta_Ca'])

# Alk Stats
r_flux_alk, p_flux_alk = get_stats(df_growth_Alk['tau_days'], df_growth_Alk['J_rem_Alk_V'])
r_delta_alk, p_delta_alk = get_stats(df_growth_Alk['tau_days'], df_growth_Alk['Delta_Alk'])

print(f"Ca Flux: n={len(df_growth_Ca)}, r={r_flux_ca:.3f}, p={p_flux_ca:.3f}")
print(f"Alk Flux: n={len(df_growth_Alk)}, r={r_flux_alk:.3f}, p={p_flux_alk:.3f}")

# ------------------------------------------------------------------
# 5. Plotting Combined Figure
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Panel A: Flux vs Tau ---
# Ca Series (Orange)
axes[0].scatter(df_growth_Ca['tau_days'], df_growth_Ca['J_rem_Ca_V'], 
                color='tab:orange', s=60, edgecolor='k', label='Ca Flux', alpha=0.8)
m_ca, c_ca = np.polyfit(df_growth_Ca['tau_days'], df_growth_Ca['J_rem_Ca_V'], 1)
axes[0].plot(df_growth_Ca['tau_days'], m_ca*df_growth_Ca['tau_days'] + c_ca, 'tab:orange', linestyle='--')

# Alk Series (Purple)
axes[0].scatter(df_growth_Alk['tau_days'], df_growth_Alk['J_rem_Alk_V'], 
                color='tab:purple', s=60, edgecolor='k', label='Alk Flux', marker='s', alpha=0.8)
m_alk, c_alk = np.polyfit(df_growth_Alk['tau_days'], df_growth_Alk['J_rem_Alk_V'], 1)
axes[0].plot(df_growth_Alk['tau_days'], m_alk*df_growth_Alk['tau_days'] + c_alk, 'tab:purple', linestyle='--')

axes[0].set_xlabel(r'Residence Time $\tau$ (days)', fontsize=12)
axes[0].set_ylabel(r'Removal Flux $J_{rem}$ (mol/day)', fontsize=12)
axes[0].set_title('Flux vs Residence Time', fontsize=12, fontweight='bold')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# Add Stats Box A
stats_text_A = (
    f"Ca (n={len(df_growth_Ca)}):\n  r={r_flux_ca:.3f}, p={p_flux_ca:.3f}\n\n"
    f"Alk (n={len(df_growth_Alk)}):\n  r={r_flux_alk:.3f}, p={p_flux_alk:.3f}"
)
axes[0].text(0.95, 0.95, stats_text_A, transform=axes[0].transAxes, 
             fontsize=10, va='top', ha='right', bbox=dict(facecolor='white', alpha=0.9))

# --- Panel B: Delta C vs Tau ---
# Ca Series (Orange)
axes[1].scatter(df_growth_Ca['tau_days'], df_growth_Ca['Delta_Ca'], 
                color='tab:orange', s=60, edgecolor='k', label='Ca Gradient', alpha=0.8)
m2_ca, c2_ca = np.polyfit(df_growth_Ca['tau_days'], df_growth_Ca['Delta_Ca'], 1)
axes[1].plot(df_growth_Ca['tau_days'], m2_ca*df_growth_Ca['tau_days'] + c2_ca, 'tab:orange', linestyle='--')

# Alk Series (Purple)
axes[1].scatter(df_growth_Alk['tau_days'], df_growth_Alk['Delta_Alk'], 
                color='tab:purple', s=60, edgecolor='k', label='Alk Gradient', marker='s', alpha=0.8)
m2_alk, c2_alk = np.polyfit(df_growth_Alk['tau_days'], df_growth_Alk['Delta_Alk'], 1)
axes[1].plot(df_growth_Alk['tau_days'], m2_alk*df_growth_Alk['tau_days'] + c2_alk, 'tab:purple', linestyle='--')

axes[1].set_xlabel(r'Residence Time $\tau$ (days)', fontsize=12)
axes[1].set_ylabel(r'Concentration Gradient $\Delta C$ (g/m$^3$)', fontsize=12)
axes[1].set_title(r'$\Delta C$ vs Residence Time', fontsize=12, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)
axes[1].legend()

# Add Stats Box B
stats_text_B = (
    f"Ca (n={len(df_growth_Ca)}):\n  r={r_delta_ca:.3f}, p={p_delta_ca:.3f}\n\n"
    f"Alk (n={len(df_growth_Alk)}):\n  r={r_delta_alk:.3f}, p={p_delta_alk:.3f}"
)
axes[1].text(0.95, 0.95, stats_text_B, transform=axes[1].transAxes, 
             fontsize=10, va='top', ha='right', bbox=dict(facecolor='white', alpha=0.9))

plt.tight_layout()
plt.savefig('Fig_S3_Correlation_Flux_ResidenceTime.png', dpi=300)
plt.show()

In [ ]:
# Cell 4 - Visualisation of formation rates

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rcParams
import numpy as np
import os
import matplotlib.dates as mdates
from scipy.interpolate import make_interp_spline

# --------------------------------------------------
# Set publication-style settings
# --------------------------------------------------
rcParams.update({
    'font.size': 12,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 10,
    'legend.title_fontsize': 10,
    'figure.figsize': (10, 12),
    'savefig.dpi': 600, # Increased to 600 for publication
    'axes.grid': True,
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'font.family': 'sans-serif',
    'mathtext.fontset': 'dejavusans'
})

# --------------------------------------------------
# Paths & constants
# --------------------------------------------------
# Paths — data files are loaded from the repo data/ directory
DATA_DIR      = os.path.join(os.path.dirname(os.path.abspath("__file__")), "data")
CONC_FILE     = os.path.join(DATA_DIR, "Full_data_for_Corbicula_Calcs.xlsx")
RESULTS_FILE  = os.path.join(DATA_DIR, "Revised_Corbicula_Mass_Balance.csv")
SUMMARY_FILE  = os.path.join(DATA_DIR, "Monthly_Water_Quality_Narrows_Summary.csv")

vline_date = pd.to_datetime('2025-05-01')

# --------------------------------------------------
# 1. Load & Smooth Historical Stats (Narrows Update)
# --------------------------------------------------
df_stats = pd.read_csv(SUMMARY_FILE)
df_stats = df_stats.set_index('Month').reindex(range(1, 13))

# Handles missing months and ensures data types are correct for interpolation
df_stats = df_stats.infer_objects(copy=False)
df_stats = df_stats.interpolate(method='linear', limit_direction='both').ffill().bfill()

# Create periodic data for wrap-around smoothing (Months 0 to 13)
# UPDATED column names to match the Narrows CSV
x_months = np.array([0] + list(range(1, 13)) + [13])

ca_m = np.concatenate(([df_stats.loc[12, 'Merged Ca_Mean']], df_stats['Merged Ca_Mean'].values, [df_stats.loc[1, 'Merged Ca_Mean']]))
ca_s = np.concatenate(([df_stats.loc[12, 'Merged Ca_SD']], df_stats['Merged Ca_SD'].values, [df_stats.loc[1, 'Merged Ca_SD']]))

alk_m = np.concatenate(([df_stats.loc[12, 'Alkalinity (mg/L) as CaCO3_Mean']], df_stats['Alkalinity (mg/L) as CaCO3_Mean'].values, [df_stats.loc[1, 'Alkalinity (mg/L) as CaCO3_Mean']]))
alk_s = np.concatenate(([df_stats.loc[12, 'Alkalinity (mg/L) as CaCO3_SD']], df_stats['Alkalinity (mg/L) as CaCO3_SD'].values, [df_stats.loc[1, 'Alkalinity (mg/L) as CaCO3_SD']]))

# Create Spline functions (k=3 for cubic)
spl_ca_m = make_interp_spline(x_months, ca_m, k=3)
spl_ca_s = make_interp_spline(x_months, ca_s, k=3)
spl_alk_m = make_interp_spline(x_months, alk_m, k=3)
spl_alk_s = make_interp_spline(x_months, alk_s, k=3)

def get_smooth_hist(dates):
    """Calculates spline value for any date based on day-of-year fraction."""
    dates_s = pd.Series(pd.to_datetime(dates))
    m_dec = dates_s.dt.month + (dates_s.dt.day - 1) / dates_s.dt.days_in_month
    return spl_ca_m(m_dec), spl_ca_s(m_dec), spl_alk_m(m_dec), spl_alk_s(m_dec)

# --------------------------------------------------
# 2. Load & Prepare Experimental Data
# --------------------------------------------------
df_conc = pd.read_excel(CONC_FILE, sheet_name='Concs and Flow')
df_conc['DateTime'] = pd.to_datetime(df_conc['Date'])
df_conc['daily_date'] = pd.to_datetime(df_conc['DateTime'].dt.date)
df_conc['Ca_mg_L'] = df_conc['Ca (ug/L)'] / 1000

df_conc_averaged = df_conc.groupby(['daily_date', 'Station', 'Depth']).agg({
    'Ca_mg_L': 'mean',
    'Alkalinity (mg/L)': 'mean',
    'Flow (cumecs, m3/s)': 'mean'
}).reset_index()

# Map smooth historical values
ca_m_v, ca_s_v, alk_m_v, alk_s_v = get_smooth_hist(df_conc_averaged['daily_date'])
df_conc_averaged['Ca_Hist_Mean'] = ca_m_v
df_conc_averaged['Ca_Hist_SD']   = ca_s_v
df_conc_averaged['Alk_Hist_Mean'] = alk_m_v
df_conc_averaged['Alk_Hist_SD']  = alk_s_v

# --------------------------------------------------
# 3. Prepare Data Structures
# --------------------------------------------------
cols_to_keep = ['daily_date', 'Ca_mg_L', 'Alkalinity (mg/L)', 'Ca_Hist_Mean', 'Ca_Hist_SD', 'Alk_Hist_Mean', 'Alk_Hist_SD']

df_ani  = df_conc_averaged.query("Station == 'Aniwaniwa Reserve' & Depth == 'Top'")[cols_to_keep].sort_values('daily_date')
df_golf = df_conc_averaged.query("Station == 'Golf Course' & Depth == 'Top'")[cols_to_keep].sort_values('daily_date')
df_ani_golf = pd.merge(df_ani, df_golf, on='daily_date', how='inner', suffixes=('_ani', '_golf'))

df_kara_top = df_conc_averaged.query("Station == 'Lake Karapiro' & Depth == 'Top'")[cols_to_keep].sort_values('daily_date')
df_kara_mid = df_conc_averaged.query("Station == 'Lake Karapiro' & Depth == 'Middle'")[cols_to_keep].sort_values('daily_date')
df_kara_bot = df_conc_averaged.query("Station == 'Lake Karapiro' & Depth == 'Bottom'")[cols_to_keep].sort_values('daily_date')

df_kara = pd.merge(df_kara_top, df_kara_bot, on='daily_date', how='inner', suffixes=('_top', '_bot'))
df_kara = pd.merge(df_kara, df_kara_mid, on='daily_date', how='inner')
df_kara = df_kara.rename(columns={'Ca_mg_L': 'Ca_mg_L_mid', 'Alkalinity (mg/L)': 'Alkalinity_Mean_mid'})

# --------------------------------------------------
# 4. Load Results
# --------------------------------------------------
df_results = pd.read_csv(RESULTS_FILE)
df_results['Date'] = pd.to_datetime(df_results['Date'])
V_LAKE = 85e6 
df_results['tau_days'] = V_LAKE / (df_results['Downstream Flow Karapiro (cumecs, m3/s)'] * 86400)

def add_season_labels(ax):
    season_starts = [(pd.Timestamp('2023-12-01'), 'Summer'), (pd.Timestamp('2024-03-01'), 'Autumn'),
                     (pd.Timestamp('2024-06-01'), 'Winter'), (pd.Timestamp('2024-09-01'), 'Spring'),
                     (pd.Timestamp('2024-12-01'), 'Summer'), (pd.Timestamp('2025-03-01'), 'Autumn'),
                     (pd.Timestamp('2025-06-01'), 'Winter'), (pd.Timestamp('2025-09-01'), 'Spring')]
    trans = ax.get_xaxis_transform()
    x_min, x_max = ax.get_xlim()
    for i in range(len(season_starts) - 1):
        s_date, label = season_starts[i]
        e_date = season_starts[i+1][0]
        s_num, e_num = mdates.date2num(s_date), mdates.date2num(e_date)
        mid_date = mdates.num2date(s_num + (e_num - s_num) / 2)
        if e_num > x_min and s_num < x_max:
            ax.text(mid_date, 1.02, label, transform=trans, ha='center', va='bottom', fontsize=12, clip_on=False)
            if s_num >= x_min:
                ax.text(s_date, 1.0, '|', transform=trans, ha='center', va='center', fontsize=12, clip_on=False)

# --------------------------------------------------
# 5. Create Figure
# --------------------------------------------------
fig, axs = plt.subplots(3, 2, figsize=(10, 12), sharex=True)

# Generate smooth background range
full_dates = pd.date_range(df_conc_averaged['daily_date'].min(), df_conc_averaged['daily_date'].max(), freq='D')
s_ca_m, s_ca_s, s_alk_m, s_alk_s = get_smooth_hist(full_dates)

# Panel A
ax = axs[0, 0]
ax.fill_between(full_dates, s_ca_m - s_ca_s, s_ca_m + s_ca_s, color='gray', alpha=0.15, label='Hist. Monthly ±1σ')
ax.plot(full_dates, s_ca_m, color='black', linestyle='--', linewidth=0.8, label='Hist. Median')
ax.plot(df_ani_golf['daily_date'], df_ani_golf['Ca_mg_L_ani'], label='ECP-S1', color='blue', marker='o', linestyle='-')
ax.plot(df_ani_golf['daily_date'], df_ani_golf['Ca_mg_L_golf'], label='ECP-S3', color='green', marker='s', linestyle='-')
ax.fill_between(df_ani_golf['daily_date'], df_ani_golf['Ca_mg_L_ani'], df_ani_golf['Ca_mg_L_golf'], where=(df_ani_golf['Ca_mg_L_ani'] > df_ani_golf['Ca_mg_L_golf']), color='blue', alpha=0.15, interpolate=True)
ax.fill_between(df_ani_golf['daily_date'], df_ani_golf['Ca_mg_L_ani'], df_ani_golf['Ca_mg_L_golf'], where=(df_ani_golf['Ca_mg_L_ani'] < df_ani_golf['Ca_mg_L_golf']), color='green', alpha=0.15, interpolate=True)
ax.axvline(vline_date, color='gray', linestyle='-', linewidth=1)
ax.set_ylabel('Ca Concentration (mg/L)')
ax.legend(loc='upper right', framealpha=1.0, borderpad=0, fontsize=8).get_frame().set_facecolor('white')
ax.set_ylim(top=8)
ax.text(-0.15, 1.05, 'A', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')
add_season_labels(ax)

# Panel B
ax = axs[0, 1]
ax.fill_between(full_dates, s_ca_m - s_ca_s, s_ca_m + s_ca_s, color='gray', alpha=0.15)
ax.plot(full_dates, s_ca_m, color='black', linestyle='--', linewidth=0.8)
ax.plot(df_kara['daily_date'], df_kara['Ca_mg_L_top'], label='2 m', color='blue', marker='o', linestyle='-')
ax.plot(df_kara['daily_date'], df_kara['Ca_mg_L_mid'], label='13 m', color='darkgreen', marker='^', linestyle='-')
ax.plot(df_kara['daily_date'], df_kara['Ca_mg_L_bot'], label='25 m', color='red', marker='s', linestyle='-')
ax.fill_between(df_kara['daily_date'], df_kara['Ca_mg_L_top'], df_kara['Ca_mg_L_bot'], where=(df_kara['Ca_mg_L_top'] > df_kara['Ca_mg_L_bot']), color='red', alpha=0.15, interpolate=True)
ax.fill_between(df_kara['daily_date'], df_kara['Ca_mg_L_top'], df_kara['Ca_mg_L_bot'], where=(df_kara['Ca_mg_L_top'] < df_kara['Ca_mg_L_bot']), color='blue', alpha=0.15, interpolate=True)
ax.axvline(vline_date, color='gray', linestyle='-', linewidth=1)
ax.set_ylabel('Ca Concentration (mg/L)')
ax.legend(loc='lower right', title='ECP-S2', framealpha=1.0, borderpad=0, fontsize=8).get_frame().set_facecolor('white')
ax.text(-0.15, 1.05, 'B', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')
add_season_labels(ax)

# Panel C
ax = axs[1, 0]
ax.fill_between(full_dates, s_alk_m - s_alk_s, s_alk_m + s_alk_s, color='gray', alpha=0.15, label='Hist. Monthly ±1σ')
ax.plot(full_dates, s_alk_m, color='black', linestyle='--', linewidth=0.8)
ax.plot(df_ani_golf['daily_date'], df_ani_golf['Alkalinity (mg/L)_ani'], label='ECP-S1', color='blue', marker='o', linestyle='-')
ax.plot(df_ani_golf['daily_date'], df_ani_golf['Alkalinity (mg/L)_golf'], label='ECP-S3', color='green', marker='s', linestyle='-')
ax.fill_between(df_ani_golf['daily_date'], df_ani_golf['Alkalinity (mg/L)_ani'], df_ani_golf['Alkalinity (mg/L)_golf'], where=(df_ani_golf['Alkalinity (mg/L)_ani'] > df_ani_golf['Alkalinity (mg/L)_golf']), color='blue', alpha=0.15, interpolate=True)
ax.fill_between(df_ani_golf['daily_date'], df_ani_golf['Alkalinity (mg/L)_ani'], df_ani_golf['Alkalinity (mg/L)_golf'], where=(df_ani_golf['Alkalinity (mg/L)_ani'] < df_ani_golf['Alkalinity (mg/L)_golf']), color='green', alpha=0.15, interpolate=True)
ax.axvline(vline_date, color='gray', linestyle='-', linewidth=1)
ax.set_ylabel('Alkalinity (mg/L as CaCO₃)')
ax.legend(loc='lower left', framealpha=1.0, borderpad=0, fontsize=8).get_frame().set_facecolor('white')
ax.set_ylim(30, 47.5)
ax.text(-0.15, 1.05, 'C', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')

# Panel D
ax = axs[1, 1]
ax.fill_between(full_dates, s_alk_m - s_alk_s, s_alk_m + s_alk_s, color='gray', alpha=0.15)
ax.plot(full_dates, s_alk_m, color='black', linestyle='--', linewidth=0.8)
ax.plot(df_kara['daily_date'], df_kara['Alkalinity (mg/L)_top'], label='2 m', color='blue', marker='o', linestyle='-')
ax.plot(df_kara['daily_date'], df_kara['Alkalinity_Mean_mid'], label='13 m', color='darkgreen', marker='^', linestyle='-')
ax.plot(df_kara['daily_date'], df_kara['Alkalinity (mg/L)_bot'], label='25 m', color='red', marker='s', linestyle='-')
ax.fill_between(df_kara['daily_date'], df_kara['Alkalinity (mg/L)_top'], df_kara['Alkalinity (mg/L)_bot'], where=(df_kara['Alkalinity (mg/L)_top'] > df_kara['Alkalinity (mg/L)_bot']), color='red', alpha=0.15, interpolate=True)
ax.fill_between(df_kara['daily_date'], df_kara['Alkalinity (mg/L)_top'], df_kara['Alkalinity (mg/L)_bot'], where=(df_kara['Alkalinity (mg/L)_top'] < df_kara['Alkalinity (mg/L)_bot']), color='blue', alpha=0.15, interpolate=True)
ax.axvline(vline_date, color='gray', linestyle='-', linewidth=1)
ax.text(vline_date - pd.Timedelta(days=10), 30.5, 'Net CaCO₃ precipitation', rotation=90, va='bottom', ha='right', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9, pad=1), fontsize=8)
ax.text(vline_date + pd.Timedelta(days=10), 30.5, 'Net CaCO₃ dissolution', rotation=90, va='bottom', ha='left', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9, pad=1), fontsize=8)
ax.set_ylabel('Alkalinity (mg/L as CaCO₃)')
ax.set_ylim(30, 47.5)
ax.text(-0.15, 1.05, 'D', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')

# Panel E
ax = axs[2, 0]
bar_width, offset_ca, offset_alk = pd.Timedelta('4D'), pd.Timedelta('-2D'), pd.Timedelta('2D')
ax.bar(df_results['Date'] + offset_ca, df_results['m_CaCO3_Ca_L'], width=bar_width, alpha=1.0, label=r'Flux from $J_{rem, Ca}$', color='tab:orange')
ax.bar(df_results['Date'] + offset_alk, df_results['m_CaCO3_Alk_L'], width=bar_width, alpha=0.7, label=r'Flux from $J_{rem, Alk}$', color='tab:purple')
ax.axvline(vline_date, color='gray', linestyle='-', linewidth=1)
ax.set_ylabel(r'Removal Flux $J_{rem, CaCO_3}$ (t/day)' + '\n' + r'(Longitudinal Depletion)')
ax.legend(loc='upper right', framealpha=1.0, borderpad=0, fontsize=9).get_frame().set_facecolor('white')
ax.set_ylim(bottom=0)
ax.text(-0.15, 1.05, 'E', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

# Panel F
ax = axs[2, 1]
ax.bar(df_results['Date'] + offset_ca, df_results['m_CaCO3_Ca_V'], width=bar_width, alpha=1.0, label=r'Flux from $J_{rem, Ca}$', color='tab:orange')
ax.bar(df_results['Date'] + offset_alk, df_results['m_CaCO3_Alk_V'], width=bar_width, alpha=0.7, label=r'Flux from $J_{rem, Alk}$', color='tab:purple')
ax.axvline(vline_date, color='gray', linestyle='-', linewidth=1)
ax.set_ylabel(r'Removal Flux $J_{rem, CaCO_3}$ (t/day)' + '\n' + r'(Vertical Depletion)')
ax2 = ax.twinx()
ax2.plot(df_results['Date'], df_results['tau_days'], color='black', linestyle='--', alpha=0.3, label='Residence Time')
ax2.set_ylabel(r'Residence Time $\tau$ (days)')
ax2.set_ylim(bottom=0, top=10)
ax2.grid(False)
l1, lab1 = ax.get_legend_handles_labels()
l2, lab2 = ax2.get_legend_handles_labels()
ax.legend(l1 + l2, lab1 + lab2, loc='upper right', framealpha=1.0, borderpad=0, fontsize=9).get_frame().set_facecolor('white')
ax.set_ylim(bottom=0)
ax.text(-0.15, 1.05, 'F', transform=ax.transAxes, fontsize=12, fontweight='bold', va='bottom')
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

plt.subplots_adjust(top=0.92, hspace=0.25)
plt.tight_layout(rect=[0, 0, 1, 0.95])

output_path = 'Fig_4_Biomineralisation_Flux.png'
plt.savefig(output_path, dpi=600, bbox_inches='tight')
plt.show()
print(f"Figure saved as: {output_path}")

In [ ]:
# Cell 5 - Compare surveyed densities to modelled populations - Box plots

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch

# Set ES&T-style formatting
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = ['Arial']
mpl.rcParams['font.size'] = 11
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['axes.titlesize'] = 12
mpl.rcParams['legend.fontsize'] = 10
mpl.rcParams['xtick.labelsize'] = 10
mpl.rcParams['ytick.labelsize'] = 10
mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['lines.linewidth'] = 0.5
mpl.rcParams['axes.linewidth'] = 0.5
mpl.rcParams['text.usetex'] = False 

# ------------------------------------------------------------------
# 2. Data Loading & Preparation
# ------------------------------------------------------------------
# Load the revised mass balance results
try:
    df_results = pd.read_csv(os.path.join('data', 'Revised_Corbicula_Mass_Balance.csv'))
except FileNotFoundError:
    # Create dummy data if file missing for testing
    np.random.seed(42)
    df_results = pd.DataFrame({
        'V_bio_Ca_V': np.abs(np.random.normal(25, 5, 100)),
        'V_bio_Alk_V': np.abs(np.random.normal(28, 6, 100))
    })
    print("WARNING: Using dummy data because CSV was not found.")

# Extract Positive Biovolume Rates (m3/day) for Residence Methods
v_bio_ca_v = df_results[df_results['V_bio_Ca_V'] > 0]['V_bio_Ca_V'].tolist()
v_bio_alk_v = df_results[df_results['V_bio_Alk_V'] > 0]['V_bio_Alk_V'].tolist()

# ------------------------------------------------------------------
# 3. Density Estimation Parameters
# ------------------------------------------------------------------
# Fixed Median Growth Rate
# 0.0055 cm3/day/ind -> Convert to m3/day/ind (1 cm3 = 1e-6 m3)
dv_dt_median_m3 = 0.0055 * 1e-6 

# Fixed Colonizable Area (<1 m depth)
AREA_COLONIZABLE = 7.41e6  # m2

# ------------------------------------------------------------------
# 4. Calculation Logic
# ------------------------------------------------------------------
def compute_daily_density(bio_m3_list, dv_dt, area):
    """
    Calculates list of daily density estimates (ind/m2).
    """
    # Population N = V_bio (m3/d) / growth_rate (m3/d/ind)
    N_list = [v / dv_dt for v in bio_m3_list]
    
    # Density D = N / Area
    density_list = [n / area for n in N_list]
    
    return density_list

# Compute Densities
dens_ca_v = compute_daily_density(v_bio_ca_v, dv_dt_median_m3, AREA_COLONIZABLE)
dens_alk_v = compute_daily_density(v_bio_alk_v, dv_dt_median_m3, AREA_COLONIZABLE)

# Combined (Pooled) Dataset for Weighted Model
combined_densities = dens_ca_v + dens_alk_v

# Calculate Statistics
model_mean = np.mean(combined_densities)
model_std = np.std(combined_densities)

# Field Survey Data (May 2025)
survey_data = {
    'Waipuke': 1134.5, 
    'Horahora': 830.3, 
    'Moana roa': 418.4, 
    'Bobs landing': 394.7, 
    'Little Waipa': 7.1
}
survey_vals = list(survey_data.values())
survey_mean = np.mean(survey_vals)
survey_std = np.std(survey_vals, ddof=1)

# Print Summary
print(f"--- Results Summary (Area < 1m: {AREA_COLONIZABLE/1e6:.3f} km2) ---")
print(f"Weighted Model Density: {model_mean:.0f} ± {model_std:.0f} ind/m2")
print(f"Survey Mean Density:    {survey_mean:.0f} ± {survey_std:.0f} ind/m2")

# ------------------------------------------------------------------
# 5. Plotting
# ------------------------------------------------------------------
fig, axs = plt.subplots(2, 1, figsize=(6, 6), constrained_layout=True)

# --- Panel A: Biovolume Rates (Vertical (Vert.) Methods Only) ---
data_a = [v_bio_ca_v, v_bio_alk_v]
labels_a = ['Vert.\nCa', 'Vert.\nAlk']

axs[0].boxplot(data_a, labels=labels_a, widths=0.5, 
               flierprops={'marker': 'o', 'markersize': 4, 'markerfacecolor': 'gray', 'markeredgecolor': 'none', 'alpha': 0.5},
               medianprops={'color': 'black'}, boxprops={'linewidth': 0.8})
axs[0].set_ylabel(r'Biovolume Formation Rate (m$^3$ day$^{-1}$)')
axs[0].set_title(r'$\mathbf{A}$', loc='left') # Removed redundant title text
axs[0].grid(True, linestyle='--', alpha=0.3)
axs[0].set_ylim(bottom=0)

# --- Panel B: Density Estimates (Vertical (Vert.) Methods Only, Area < 1m) ---
data_b = [
    dens_ca_v,
    dens_alk_v,
    combined_densities
]

labels_b = [
    'Vert. Ca\n(<1m)', 
    'Vert. Alk\n(<1m)',
    'Combined\nModel'
]

# Create Boxplots
bp = axs[1].boxplot(data_b, labels=labels_b, widths=0.5, positions=[1, 2, 3],
                    flierprops={'marker': 'o', 'markersize': 4, 'markerfacecolor': 'gray', 'markeredgecolor': 'none', 'alpha': 0.5},
                    medianprops={'color': 'black'}, boxprops={'linewidth': 0.8})

# Add Weighted Model Mean (Blue Bar) on top of Combined Boxplot
pos_model = 3
axs[1].add_patch(mpl.patches.Rectangle((pos_model - 0.25, model_mean - model_std), 0.5, 2 * model_std, 
                                       color='blue', alpha=0.2, linewidth=0))
axs[1].plot([pos_model - 0.25, pos_model + 0.25], [model_mean, model_mean], color='blue', lw=2)

# Add Survey Mean (Green Bar) separate position
pos_survey = 4
axs[1].boxplot([], positions=[pos_survey]) # Empty placeholder to extend x-axis
axs[1].add_patch(mpl.patches.Rectangle((pos_survey - 0.25, survey_mean - survey_std), 0.5, 2 * survey_std, 
                                       color='green', alpha=0.2, linewidth=0))
axs[1].plot([pos_survey - 0.25, pos_survey + 0.25], [survey_mean, survey_mean], color='green', lw=2)

# Update X-Axis Labels
final_labels = labels_b + ['Survey\nAvg']
axs[1].set_xticks([1, 2, 3, 4])
axs[1].set_xticklabels(final_labels, rotation=0, ha='center')

# Formatting
axs[1].set_ylabel(r'Estimated Density (ind m$^{-2}$)')
axs[1].set_title(r'$\mathbf{B}$', loc='left') # Removed redundant title text
axs[1].grid(True, linestyle='--', alpha=0.3)
axs[1].set_ylim(bottom=0)

# Add Survey Reference Lines
colors = ['red', 'blue', 'green', 'orange', 'purple']
for i, (site, val) in enumerate(survey_data.items()):
    axs[1].axhline(y=val, color=colors[i], linestyle='--', linewidth=0.8, alpha=0.7, label=f'{site}')

# Legend for survey lines
axs[1].legend(title='Survey Sites', loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0, frameon=False)

# Save
plt.savefig('Fig2_Final_Density_Vert_Only_Cleaned.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure saved as 'Fig_5_P_Density.png'")